# Draw Choice

Each turn, we choose the known top of the discard pile or a card from stock, then discard back to 10. We value each option by the best position it leads to.

In [1]:
import sys, os, time, random
sys.path.insert(0, os.path.abspath('../..'))
from itertools import combinations

from agent.cards import Card, SUITS, RANKS, RANK_INDEX, make_deck, find_best_melds
from agent.inference import BeliefState
from agent.eval.sim import _should_draw_discard   # the greedy baseline we must reduce to

def deadwood(cards):
    return sum(c.value for c in cards)

## 1. Valuing a draw.

We use the same machinery as with the discard policy, $D(d)$ being the our score if we discard $d$, and $R(d)$ the infered risk discarding a card poses. Value function:

$$\text{V(d)}=D(d)+\alpha R(d)$$

In [2]:
def melds_containing(d):
    # =3-card melds d could complete, as (partner, partner) pairs.
    melds = [(x, y) for x, y in combinations([Card(d.rank, s) for s in SUITS if s != d.suit], 2)]
    ri = RANK_INDEX[d.rank]
    for start in (ri - 2, ri - 1, ri):
        if start < 0 or start + 2 >= len(RANKS):
            continue
        run = [Card(RANKS[start + k], d.suit) for k in range(3)]
        melds.append(tuple(c for c in run if c != d))
    return melds

def keep_term(d, hand):
    # D(d): our deadwood after discarding d.
    _, dw = find_best_melds([c for c in hand if c != d])
    return deadwood(dw)

def risk_term(d, bs):
    # R(d): expected points conceded if the opponent melds d (independence approx).
    return sum(bs.prob(x) * bs.prob(y) * (d.value + x.value + y.value) for x, y in melds_containing(d))

def position_cost(hand, bs, alpha, forbidden=None):
    # Best discard score reachable from an 11-card hand: min_d D(d)+alpha*R(d).
    # `forbidden` excludes a card from being discarded (the no-rediscard rule).
    best = (float('inf'), None)
    for d in hand:
        if d == forbidden:
            continue
        c = keep_term(d, hand) + alpha * risk_term(d, bs)
        if c < best[0]:
            best = (c, d)
    return best   # (cost, discard)

def hand_deadwood(hand):
    # Deadwood of the current 10-card hand
    _, dw = find_best_melds(hand)
    return deadwood(dw)

## 2. Taking the discard top $t$

Taking $t$ is deterministic. $V_\text{take}=\texttt{position\_cost}(\text{hand}\cup\{t\})$ with the inner $\min_d$ **excluding** $d=t$ (gin forbids re-discarding the card you just drew). The benefit of taking is $B(t)=\text{(current deadwood)} - V_\text{take}$. Scenario below: the opponent throws 9♥, which extends our 7♥8♥ — taking it should give a large positive $B$.

In [3]:
# We hold 7H 8H, so our outs are 6H and 9H. The opponent is visibly collecting
# hearts: they took 4H off the discard (upweighting low hearts), then threw 9H --
# useful to us, outside their 2..6 window. So t = 9H is a tempting take, yet 6H
# (our other out) is now probably in their hand.
hand = [Card('7','H'), Card('8','H'), Card('2','C'), Card('5','D'), Card('9','C'),
        Card('K','S'), Card('Q','D'), Card('4','S'), Card('J','H'), Card('3','C')]
bs = BeliefState(list(hand), Card('4','H'))                       # face-up = 4H
bs.observe_opponent_draw_discard_bayesian(Card('4','H'), nu=3.0)  # opp grabs 4H -> hearts upweighted
bs.observe_opponent_discard(Card('9','H'))                        # opp throws 9H (now the top)
t = Card('9','H')

alpha = 0.1
cur = hand_deadwood(hand)
Vtake, dt = position_cost(hand + [t], bs, alpha, forbidden=t)
print(f'belief sum (invariant): {bs.hand_size_belief:.3f}   stock size: {bs.stock_size}')
print(f'current deadwood       : {cur}')
print(f'V_take({t})            : {Vtake:.2f}   (then discard {dt})')
print(f'benefit B(t)           : {cur - Vtake:.2f}')

belief sum (invariant): 10.000   stock size: 31
current deadwood       : 68
V_take(9H)            : 43.36   (then discard KS)
benefit B(t)           : 24.64


## 3. The Stock Draw as an Expectation

A stock draw is a lottery over the $S$ unseen cards. Using the belief state, the top stock card is $c$ with probability $(1-p(c))/S$ over cards with $0<p(c)<1$.

$$V_\text{stock}=\sum_c P(\text{top}=c)\,V(\text{draw }c)$$

In [8]:
def stock_candidates(bs):
    # Cards that could still be in the stock (0 < p < 1).
    return [c for c in make_deck() if 0.0 < bs.prob(c) < 1.0]

cand = stock_candidates(bs)
# P(top = c) is proportional to (1 - p(c)); normalise by Z = sum(1 - p).
# In a consistent game Z equals the stock size exactly.
Z = sum(1 - bs.prob(c) for c in cand)
p_top = {c: (1 - bs.prob(c)) / Z for c in cand}
print(f'stock candidates: {len(cand)}   Z = {Z:.2f}   stock_size = {bs.stock_size}   sum P(top): {sum(p_top.values()):.6f}')

# Belief skews the lottery: cards the opponent likely holds are unlikely to surface.
print('\nleast likely from stock (opponent probably holds them):')
for c in sorted(cand, key=lambda c: bs.prob(c), reverse=True)[:5]:
    print(f'  {c}:  p(opp)={bs.prob(c):.2f}  ->  P(top)={p_top[c]:.4f}')

t0 = time.time()
Vstock = sum(p * position_cost(hand + [c], bs, alpha)[0] for c, p in p_top.items())
print(f'\nV_stock = {Vstock:.2f}')
print(f'V_take  = {Vtake:.2f}   ->  before info penalty, {"TAKE" if Vtake < Vstock else "STOCK"} is cheaper')

stock candidates: 40   Z = 31.00   stock_size = 31   sum P(top): 1.000000

least likely from stock (opponent probably holds them):
  2H:  p(opp)=0.52  ->  P(top)=0.0155
  3H:  p(opp)=0.52  ->  P(top)=0.0155
  5H:  p(opp)=0.52  ->  P(top)=0.0155
  6H:  p(opp)=0.52  ->  P(top)=0.0155
  4D:  p(opp)=0.52  ->  P(top)=0.0155

V_stock = 64.33
V_take  = 43.36   ->  before info penalty, TAKE is cheaper


## 4. Belief Weighting Vs Greedy.

Compare the belief-weighted $V_\text{stock}$ against a uniform average over the same candidates. Here our out-card $6\heartsuit$ is probably in the opponent's hand, so belief should judge the stock slightly worse than a naive uniform average (which over-credits our chance of drawing $6\heartsuit$).

In [9]:
Vstock_belief  = sum(p * position_cost(hand + [c], bs, alpha)[0] for c, p in p_top.items())
Vstock_uniform = sum(position_cost(hand + [c], bs, alpha)[0] for c in cand) / len(cand)
print(f'V_stock (belief-weighted): {Vstock_belief:.3f}')
print(f'V_stock (uniform)        : {Vstock_uniform:.3f}')
print(f'difference               : {Vstock_belief - Vstock_uniform:+.3f}')

V_stock (belief-weighted): 64.329
V_stock (uniform)        : 63.892
difference               : +0.438


## 5. Information Leakage Penalty

Taking face-up tells the opponent $t$ is useful to us (they upweight $t$'s partners, our own Bayesian signal in reverse). Price it as a penalty added to $V_\text{take}$; here a simple proxy $\gamma\cdot\text{value}(t)$. Below is a sweep of $\gamma$.

In [10]:
print(f'V_take = {Vtake:.2f}   V_stock = {Vstock:.2f}\n')
for gamma in [0.0, 0.5, 1.0, 2.0, 4.0]:
    penalty = gamma * t.value
    decision = 'TAKE discard' if (Vtake + penalty) < Vstock else 'DRAW stock'
    print(f'gamma={gamma:>3}:  V_take + {penalty:>4.1f} = {Vtake + penalty:>5.2f}  vs  {Vstock:.2f}   ->  {decision}')

V_take = 43.36   V_stock = 64.33

gamma=0.0:  V_take +  0.0 = 43.36  vs  64.33   ->  TAKE discard
gamma=0.5:  V_take +  4.5 = 47.86  vs  64.33   ->  TAKE discard
gamma=1.0:  V_take +  9.0 = 52.36  vs  64.33   ->  TAKE discard
gamma=2.0:  V_take + 18.0 = 61.36  vs  64.33   ->  TAKE discard
gamma=4.0:  V_take + 36.0 = 79.36  vs  64.33   ->  DRAW stock


## 6. Reduction to the Greedy Baseline

If we strip the belief weighting and the info penalty ($\alpha=0$, $\gamma=0$) and use the do-nothing baseline (take iff the draw beats current deadwood). This should match `_should_draw_discard` from `sim.py` exactly.

In [11]:
random.seed(5)
N = 300
agree = 0
for _ in range(N):
    deck = make_deck(); random.shuffle(deck)
    h = deck[:10]; fu = deck[10]
    b = BeliefState(list(h), fu)
    vt, _ = position_cost(h + [fu], b, alpha=0.0)          # greedy: rediscard allowed
    ours = vt < hand_deadwood(h)                            # take iff it lowers deadwood
    if ours == _should_draw_discard(h, fu):
        agree += 1
print(f'ours(alpha=0) agrees with greedy _should_draw_discard: {agree}/{N}')

ours(alpha=0) agrees with greedy _should_draw_discard: 300/300


## Summary

The draw choice follows sensibly from the discard policy: we value each option by the best position it reaches and compare the known take against the belief-weighted stock expectation. The weak point is the info-penalty proxy $\gamma\cdot\text{value}(t)$ — by far the most qualitative thing we've implemented. A card's value isn't really what determines the leak from taking it face-up; what matters is how many of its partners we're still collecting and how exposed taking it leaves us. Value is also mis-signed in the case that matters most — a high-value card completing a meld leaks almost nothing, yet gets the biggest penalty. The better proxy counts $t$'s live partners and vanishes when $t$ closes a meld; the eval harness can pin down the weight.